In [ ]:
from google.colab import drive
drive.mount("/content/drive")

# !unzip -q "/content/drive/MyDrive/Data/chunk_outputs_finals.zip" -d "/content"
# !unzip -q "/content/drive/MyDrive/Data/chunk_outputs1_finals.zip" -d "/content"

Mounted at /content/drive


In [ ]:
import json, numpy as np

IN_FILE  = "/content/drive/MyDrive/Data/distill_candidates.jsonl"
OUT_FILE = "/content/drive/MyDrive/Data/distill_candidates_logit.jsonl"
EPS = 1e-6   # tránh log(0) và chia 0 khi p=0 hoặc p=1

def sigmoid_to_logit(p):
    p = min(max(p, EPS), 1 - EPS)     # clip về (0,1) để không vô cực
    return np.log(p / (1 - p))

rows, all_logits = [], []
with open(IN_FILE, encoding="utf-8") as f:
    for line in f:
        if not line.strip(): continue
        r = json.loads(line)
        for c in r["candidates"]:
            c["logit"] = float(sigmoid_to_logit(c["logit"]))   # ghi đè bằng logit thật
            all_logits.append(c["logit"])
        rows.append(r)

with open(OUT_FILE, "w", encoding="utf-8") as f:
    for r in rows:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

v = np.array(all_logits)
print(f"Converted {len(rows)} query -> {OUT_FILE}")
print(f"Logit: min={v.min():.2f}, max={v.max():.2f}, mean={v.mean():.2f}")
print(f"trong [0,1]: {np.mean((v>=0)&(v<=1))*100:.1f}%  (phải THẤP giờ)")

Converted 634 query -> /content/drive/MyDrive/Data/distill_candidates_logit.jsonl
Logit: min=-11.04, max=10.58, mean=-3.77
trong [0,1]: 5.2%  (phải THẤP giờ)


In [ ]:
import json, gc, random
from pathlib import Path
from collections import defaultdict

import torch
from torch.utils.data import DataLoader
from sentence_transformers import SentenceTransformer, InputExample
from sentence_transformers.losses import MultipleNegativesRankingLoss, MatryoshkaLoss

TRAIN_FILE = "/content/drive/MyDrive/Data/train_query_positive_merged_with_gen.jsonl"
BASE_MODEL = "AITeamVN/Vietnamese_Embedding_v2"
OUT_DIR = "/content/drive/MyDrive/Data/archive/outputs/embed_clean_baseline_v2"
CKPT_NAME  = "embed_clean_mnr_1stage_seed42_v2"

CFG = {
    "epochs": 2,
    "lr": 2e-6,
    "batch": 16,
    "warmup": 10,
    "max_seq_len": 1024,
    "mrl_dims": [256, 512, 1024],
    "use_amp": True,
    "seed": 42,
}

/tmp/ipykernel_872/1755520092.py:8: DeprecationWarning: Importing from 'sentence_transformers.losses' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.losses' instead.
  from sentence_transformers.losses import MultipleNegativesRankingLoss, MatryoshkaLoss


In [ ]:
def set_seed(seed):
    random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

def load_grouped(path):
    """Gộp positive theo query -> {query: [pos, ...]}"""
    by_q = defaultdict(list)
    with open(path, encoding="utf-8") as f:
        for line in f:
            if not line.strip(): continue
            r = json.loads(line)
            q, pos = str(r.get("query","")).strip(), str(r.get("positive","")).strip()
            if q and pos: by_q[q].append(pos)
    total = sum(len(v) for v in by_q.values())
    print(f"Grouped: {len(by_q)} unique query | {total} tổng cặp "
          f"| multi-pos: {sum(len(v)>1 for v in by_q.values())} query")
    return by_q

def make_examples(by_q, epoch_seed):
    """1 positive/query cho epoch này, chọn theo seed -> mỗi epoch thấy positive khác."""
    rng = random.Random(epoch_seed)
    ex = [InputExample(texts=[q, rng.choice(poss)]) for q, poss in by_q.items()]
    rng.shuffle(ex)
    return ex

def train_embedding_one_stage(train_file=TRAIN_FILE, use_gist=False):
    set_seed(CFG["seed"])
    by_q = load_grouped(train_file)

    model = SentenceTransformer(BASE_MODEL)
    model.max_seq_length = CFG["max_seq_len"]

    guide = None
    if use_gist:
        from sentence_transformers.losses import GISTEmbedLoss
        guide = SentenceTransformer(GUIDE_MODEL); guide.max_seq_length = CFG["max_seq_len"]
        base_loss = GISTEmbedLoss(model=model, guide=guide, temperature=0.01)
    else:
        base_loss = MultipleNegativesRankingLoss(model=model, scale=20.0)

    loss = MatryoshkaLoss(model=model, loss=base_loss, matryoshka_dims=CFG["mrl_dims"])
    ckpt = str(Path(OUT_DIR) / "checkpoints" / CKPT_NAME)

    print(f"seq={CFG['max_seq_len']} | loss={'GIST' if use_gist else 'MNR'} "
          f"| lr={CFG['lr']:.0e} | batch={CFG['batch']} | epochs={CFG['epochs']}")

    # Train từng epoch riêng: mỗi epoch resample 1 positive/query khác nhau
    for epoch in range(CFG["epochs"]):
        examples = make_examples(by_q, epoch_seed=CFG["seed"] + epoch)
        loader = DataLoader(examples, batch_size=CFG["batch"], shuffle=True)
        print(f"  Epoch {epoch+1}/{CFG['epochs']} | {len(examples)} examples "
              f"| {len(loader)} batches")
        model.fit(
            train_objectives=[(loader, loss)],
            epochs=1,
            warmup_steps=CFG["warmup"] if epoch == 0 else 0,
            optimizer_params={"lr": CFG["lr"]},
            weight_decay=0.01,
            max_grad_norm=1.0,
            use_amp=CFG["use_amp"] and torch.cuda.is_available(),
            output_path=None,           # lưu 1 lần ở cuối
            save_best_model=False,
            show_progress_bar=True,
        )

    model.save(ckpt)
    del model
    if guide is not None: del guide
    gc.collect(); torch.cuda.empty_cache()
    print(f"Saved: {ckpt} | Free: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB")
    return ckpt

CKPT = train_embedding_one_stage(use_gist=False)
print("DONE:", CKPT)

Grouped: 2594 unique query | 3146 tổng cặp | multi-pos: 348 query


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/171 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.45k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

...

Step,Training Loss


  Epoch 2/2 | 2594 examples | 163 batches


Step,Training Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved: /content/drive/MyDrive/Data/archive/outputs/embed_clean_baseline_v2/checkpoints/embed_clean_mnr_1stage_seed42_v2 | Free: 82.2 GB
DONE: /content/drive/MyDrive/Data/archive/outputs/embed_clean_baseline_v2/checkpoints/embed_clean_mnr_1stage_seed42_v2


In [ ]:
import json, gc
from collections import defaultdict
import torch
from sentence_transformers import SentenceTransformer
from sentence_transformers.evaluation import InformationRetrievalEvaluator

# ---------- ĐIỀU CHỈNH ĐƯỜNG DẪN + FIELD ----------
EVAL_PATH   = "/content/drive/MyDrive/Data/eval_queries.jsonl"
CORPUS_PATH = "/content/drive/MyDrive/Data/corpus_full.jsonl"

F_QID, F_QUERY, F_GOLD, F_INTENT = "qid", "query", "gold_chunk_ids", "intent"
C_ID, C_TEXT = "chunk_id", "text"
DIMS = [256, 512, 1024]
MAX_SEQ = 1024        # PHẢI khớp lúc train (đừng eval ở 512 nếu train 1024)
# --------------------------------------------------

# 1) Corpus đầy đủ (2317) — dùng nguyên cho MỌI query (distractor thật)
corpus = {}
with open(CORPUS_PATH, encoding="utf-8") as f:
    for line in f:
        if line.strip():
            r = json.loads(line); corpus[r[C_ID]] = r[C_TEXT]
print(f"Corpus: {len(corpus)} chunks")

# 2) Eval queries + gold + intent
queries, relevant, intent_of = {}, {}, {}
with open(EVAL_PATH, encoding="utf-8") as f:
    for line in f:
        if not line.strip(): continue
        r = json.loads(line)
        qid = str(r[F_QID])
        gold = set(r.get(F_GOLD) or [])
        gold = {g for g in gold if g in corpus}          # chỉ giữ gold có trong corpus
        if not gold: continue
        queries[qid]   = r[F_QUERY]
        relevant[qid]  = gold
        intent_of[qid] = r.get(F_INTENT, "unknown")

by_intent = defaultdict(list)
for qid, it in intent_of.items(): by_intent[it].append(qid)
print(f"Eval: {len(queries)} query | intent: " +
      ", ".join(f"{k}={len(v)}" for k,v in by_intent.items()))

def make_evaluator(qids, name):
    return InformationRetrievalEvaluator(
        queries={q: queries[q] for q in qids},
        corpus=corpus,                                   # LUÔN full corpus
        relevant_docs={q: relevant[q] for q in qids},
        accuracy_at_k=[1, 5, 10], mrr_at_k=[10], ndcg_at_k=[10],
        batch_size=64, name=name, show_progress_bar=False,
    )

def run_block(model, title, qids, tag):
    n = len(qids)
    warn = "  ⚠️ n<30, tham khảo" if n < 30 else ""
    print(f"\n{'='*66}\n{title} ({n} query){warn}\n{'='*66}")
    print(f"{'Dim':>5} {'Acc@1':>8} {'Acc@5':>8} {'MRR@10':>8} {'NDCG@10':>9}")
    print("-"*50)
    ev = make_evaluator(qids, tag)
    for dim in DIMS:
        ev.truncate_dim = dim
        s = ev(model)
        p = f"{tag}_{dim}" if False else tag   # metric key dùng name của evaluator
        a1  = s[f"{tag}_cosine_accuracy@1"]
        a5  = s[f"{tag}_cosine_accuracy@5"]
        mrr = s[f"{tag}_cosine_mrr@10"]
        nd  = s[f"{tag}_cosine_ndcg@10"]
        print(f"{dim:>5} {a1*100:>7.2f}% {a5*100:>7.2f}% {mrr:>8.4f} {nd:>9.4f}")

def bench_model(model_path):
    model = SentenceTransformer(model_path)
    model.max_seq_length = MAX_SEQ
    print(f"\n########## {model_path} ##########")
    run_block(model, "TỔNG", list(queries.keys()), "all")
    for it, qids in by_intent.items():
        run_block(model, f"INTENT = {it}", qids, f"it_{it}")
    del model; gc.collect(); torch.cuda.empty_cache()

# Chạy:
bench_model("/content/drive/MyDrive/Data/outputs/embed_distill/checkpoints/embed_distill_marginmse_seed42")          # checkpoint vừa train
# bench_model(BASE_MODEL)  # bỏ comment để so với base VN_v2

Corpus: 2317 chunks
Eval: 158 query | intent: tra_cuu=148, tinh_toan=10


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]


########## /content/drive/MyDrive/Data/outputs/embed_distill/checkpoints/embed_distill_marginmse_seed42 ##########

TỔNG (158 query)
  Dim    Acc@1    Acc@5   MRR@10   NDCG@10
--------------------------------------------------
  256   62.03%   84.81%   0.7177    0.7207
  512   65.19%   88.61%   0.7561    0.7588
 1024   68.99%   90.51%   0.7813    0.7836

INTENT = tra_cuu (148 query)
  Dim    Acc@1    Acc@5   MRR@10   NDCG@10
--------------------------------------------------


KeyboardInterrupt: 

In [ ]:
import json, math, gc
import numpy as np
import torch
from pathlib import Path
from tqdm.auto import tqdm
from sentence_transformers import SentenceTransformer


# ===== Paths =====
MODEL_PATH = "/content/drive/MyDrive/Data/archive/outputs/embed_clean_baseline_v2/checkpoints/embed_clean_mnr_1stage_seed42_v2"
QUESTION_JSON = "/content/drive/MyDrive/Data/question.json"
CHUNK_DIR = "/content/drive/MyDrive/Data/archive/chunk_outputs1_finals"

DIMS = [256, 512, 1024]
BATCH_SIZE = 64

def build_corpus(chunk_dir):
    corpus = {}

    for doc_dir in Path(chunk_dir).iterdir():
        if not doc_dir.is_dir():
            continue

        doc_scope = doc_dir.name

        for cf in doc_dir.glob("*.json"):
            recs = json.load(open(cf, encoding="utf-8"))
            if not isinstance(recs, list):
                continue

            for i, rec in enumerate(recs):
                text = str(rec.get("page_content", "")).strip()
                if not text:
                    continue

                md = rec.get("metadata", {}) or {}
                raw = str(md.get("chunk_id") or f"chunk::{md.get('chunk_index', i)}").strip()
                if not raw:
                    continue

                cid = raw if raw.startswith(f"{doc_scope}::") else f"{doc_scope}::{raw}"
                corpus[cid] = text

    return corpus


def load_question_json(path, corpus):
    rows = []

    data = json.load(open(path, encoding="utf-8"))

    for i, q in enumerate(data):
        question = str(q.get("question", "")).strip()
        gold = set(q.get("gold_chunk_ids", [])) & set(corpus.keys())

        if not question or not gold:
            continue

        rows.append({
            "idx": q.get("index", i),
            "question": question,
            "gold": gold,
        })

    return rows


def encode_truncated(model, texts, dim, batch_size=64):
    emb = model.encode(
        texts,
        batch_size=batch_size,
        normalize_embeddings=False,
        convert_to_numpy=True,
        show_progress_bar=True,
    )

    if emb.shape[1] < dim:
        raise ValueError(f"Model output dim={emb.shape[1]} < requested dim={dim}")

    emb = emb[:, :dim]
    emb = emb / np.maximum(np.linalg.norm(emb, axis=1, keepdims=True), 1e-12)
    return emb.astype(np.float32)


def dcg_at_k(rels, k):
    return sum(rel / math.log2(i + 2) for i, rel in enumerate(rels[:k]))


def metrics_one(ranked_ids, gold_set, k_recall=5, k_mrr=10, k_ndcg=10):
    gold_set = set(gold_set)

    hit1 = float(ranked_ids[0] in gold_set)

    recall5 = float(any(cid in gold_set for cid in ranked_ids[:k_recall]))

    mrr10 = 0.0
    for rank, cid in enumerate(ranked_ids[:k_mrr], start=1):
        if cid in gold_set:
            mrr10 = 1.0 / rank
            break

    rels = [1.0 if cid in gold_set else 0.0 for cid in ranked_ids[:k_ndcg]]
    dcg = dcg_at_k(rels, k_ndcg)

    ideal_n = min(len(gold_set), k_ndcg)
    idcg = dcg_at_k([1.0] * ideal_n + [0.0] * (k_ndcg - ideal_n), k_ndcg)
    ndcg10 = dcg / idcg if idcg > 0 else 0.0

    return hit1, recall5, mrr10, ndcg10


def benchmark_embedding_question_json(
    model_path=MODEL_PATH,
    question_json=QUESTION_JSON,
    chunk_dir=CHUNK_DIR,
    dims=DIMS,
    batch_size=BATCH_SIZE,
):
    corpus = build_corpus(chunk_dir)
    cids = list(corpus.keys())
    docs = list(corpus.values())

    queries = load_question_json(question_json, corpus)

    print("Model:", model_path)
    print("Questions:", len(queries))
    print("Corpus:", len(corpus))

    model = SentenceTransformer(model_path)
    model.max_seq_length = 512

    results = {}

    for dim in dims:
        print("\n" + "=" * 80)
        print(f"Benchmark dim={dim}")
        print("=" * 80)

        doc_emb = encode_truncated(model, docs, dim=dim, batch_size=batch_size)
        query_emb = encode_truncated(
            model,
            [x["question"] for x in queries],
            dim=dim,
            batch_size=batch_size,
        )

        metrics = []
        rows = []

        for qi, q in enumerate(tqdm(queries, desc=f"ranking dim={dim}")):
            sims = doc_emb @ query_emb[qi]
            order = np.argsort(-sims)

            ranked_ids = [cids[j] for j in order[:10]]
            m = metrics_one(ranked_ids, q["gold"])
            metrics.append(m)

            gold_rank = None
            for rank, j in enumerate(order, start=1):
                if cids[j] in q["gold"]:
                    gold_rank = rank
                    break

            rows.append({
                "idx": q["idx"],
                "question": q["question"],
                "gold": sorted(q["gold"]),
                "top1": cids[order[0]],
                "gold_rank": gold_rank,
                "hit1": m[0],
                "recall5": m[1],
                "mrr10": m[2],
                "ndcg10": m[3],
                "top10": ranked_ids,
                "top10_scores": [float(sims[j]) for j in order[:10]],
            })

        result = {
            "n": len(metrics),
            "Hit@1": float(np.mean([m[0] for m in metrics])),
            "Recall@5": float(np.mean([m[1] for m in metrics])),
            "MRR@10": float(np.mean([m[2] for m in metrics])),
            "NDCG@10": float(np.mean([m[3] for m in metrics])),
        }

        results[dim] = result

        print(f"Hit@1:    {result['Hit@1']:.4f}")
        print(f"Recall@5: {result['Recall@5']:.4f}")
        print(f"MRR@10:   {result['MRR@10']:.4f}")
        print(f"NDCG@10:  {result['NDCG@10']:.4f}")

    del model
    gc.collect()
    torch.cuda.empty_cache()

    return results


results_question = benchmark_embedding_question_json()
results_question

Model: /content/drive/MyDrive/Data/archive/outputs/embed_clean_baseline_v2/checkpoints/embed_clean_mnr_1stage_seed42_v2
Questions: 390
Corpus: 813


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]


Benchmark dim=256


Batches:   0%|          | 0/13 [00:00<?, ?it/s]

Batches:   0%|          | 0/7 [00:00<?, ?it/s]

...

Batches:   0%|          | 0/13 [00:00<?, ?it/s]

Batches:   0%|          | 0/7 [00:00<?, ?it/s]

ranking dim=1024:   0%|          | 0/390 [00:00<?, ?it/s]

Hit@1:    0.6923
Recall@5: 0.9256
MRR@10:   0.7964
NDCG@10:  0.8377


{256: {'n': 390,
  'Hit@1': 0.6743589743589744,
  'Recall@5': 0.9025641025641026,
  'MRR@10': 0.7721347171347172,
  'NDCG@10': 0.8116914025857199},
 512: {'n': 390,
  'Hit@1': 0.6897435897435897,
  'Recall@5': 0.9179487179487179,
  'MRR@10': 0.7869240944240944,
  'NDCG@10': 0.8266888683284592},
 1024: {'n': 390,
  'Hit@1': 0.6923076923076923,
  'Recall@5': 0.9256410256410257,
  'MRR@10': 0.7963980463980463,
  'NDCG@10': 0.8377168753090499}}

In [ ]:
MODELS_TO_RUN = {
    "BGE-M3 raw": "BAAI/bge-m3",
    "Vietnamese_Embedding_v2 raw": "AITeamVN/Vietnamese_Embedding_v2",
    # "Baseline trained": "outputs/embed_clean_baseline/checkpoints/embed_clean_mnr_1stage_seed42",
    "Vietnamese_Embedding_v1 raw": "AITeamVN/Vietnamese_Embedding",
}

all_results = {}

for name, path in MODELS_TO_RUN.items():
    print("\n" + "#" * 100)
    print(name)
    print(path)
    print("#" * 100)

    try:
        all_results[name] = benchmark_embedding_question_json(
            model_path=path,
            question_json=QUESTION_JSON,
            chunk_dir=CHUNK_DIR,
            dims=DIMS,
            batch_size=BATCH_SIZE,
        )
    except Exception as e:
        print(f"FAILED {name}: {e}")


print("\n" + "=" * 100)
print(f"{'Model':<32} {'Dim':>6} {'Hit@1':>8} {'Recall@5':>10} {'MRR@10':>9} {'NDCG@10':>10}")
print("=" * 100)

for name, dim_results in all_results.items():
    for dim, r in dim_results.items():
        print(
            f"{name:<32} {dim:>6} "
            f"{r['Hit@1'] * 100:>7.2f}% "
            f"{r['Recall@5'] * 100:>9.2f}% "
            f"{r['MRR@10']:>9.4f} "
            f"{r['NDCG@10']:>10.4f}"
        )


####################################################################################################
BGE-M3 raw
BAAI/bge-m3
####################################################################################################
Model: BAAI/bge-m3
Questions: 390
Corpus: 813


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

...

Hit@1:    0.6718
Recall@5: 0.8795
MRR@10:   0.7642
NDCG@10:  0.8061

Benchmark dim=1024


Batches:   0%|          | 0/13 [00:00<?, ?it/s]

Batches:   0%|          | 0/7 [00:00<?, ?it/s]

ranking dim=1024:   0%|          | 0/390 [00:00<?, ?it/s]

Hit@1:    0.7000
Recall@5: 0.9077
MRR@10:   0.7890
NDCG@10:  0.8305

Model                               Dim    Hit@1   Recall@5    MRR@10    NDCG@10
BGE-M3 raw                          256   62.82%     85.13%    0.7221     0.7632
BGE-M3 raw                          512   66.15%     89.74%    0.7534     0.7922
BGE-M3 raw                         1024   67.95%     91.28%    0.7748     0.8148
Vietnamese_Embedding_v2 raw         256   63.08%     87.18%    0.7310     0.7727
Vietnamese_Embedding_v2 raw         512   70.00%     89.74%    0.7827     0.8183
Vietnamese_Embedding_v2 raw        1024   70.51%     91.79%    0.7966     0.8329
Vietnamese_Embedding_v1 raw         256   61.54%     84.36%    0.7201     0.7649
Vietnamese_Embedding_v1 raw         512   67.18%     87.95%    0.7642     0.8061
Vietnamese_Embedding_v1 raw        1024   70.00%     90.77%    0.7890     0.8305


In [ ]:
import json, math, gc
import numpy as np
import torch
from pathlib import Path
from tqdm.auto import tqdm
from sentence_transformers import SentenceTransformer


# ===== Internal clean eval =====
EVAL_QUERIES = "/content/drive/MyDrive/Data/eval_queries.jsonl"
CORPUS_FULL = "/content/drive/MyDrive/Data/corpus_full.jsonl"

MODELS_TO_RUN = {
    "BGE-M3 raw": "BAAI/bge-m3",
    "Vietnamese_Embedding_v2 raw": "AITeamVN/Vietnamese_Embedding_v2",
    "Baseline trained": "outputs/embed_clean_baseline/checkpoints/embed_clean_mnr_1stage_seed42",
    "Vietnamese_Embedding_v1 raw": "AITeamVN/Vietnamese_Embedding",
}

DIMS = [256, 512, 1024]
BATCH_SIZE = 64


def load_internal_eval(eval_path, corpus_path):
    corpus = {}
    with open(corpus_path, encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            rec = json.loads(line)
            corpus[rec["chunk_id"]] = rec["text"]

    queries = []
    with open(eval_path, encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            rec = json.loads(line)
            gold = set(rec.get("gold_chunk_ids", [])) & set(corpus.keys())
            if not gold:
                continue
            queries.append({
                "qid": rec["qid"],
                "question": rec["query"],
                "intent": rec.get("intent"),
                "gold": gold,
            })

    return queries, corpus


def encode_truncated(model, texts, dim, batch_size=64):
    emb = model.encode(
        texts,
        batch_size=batch_size,
        normalize_embeddings=False,
        convert_to_numpy=True,
        show_progress_bar=True,
    )

    real_dim = emb.shape[1]
    use_dim = min(dim, real_dim)

    emb = emb[:, :use_dim]
    emb = emb / np.maximum(np.linalg.norm(emb, axis=1, keepdims=True), 1e-12)
    return emb.astype(np.float32), real_dim, use_dim


def dcg_at_k(rels, k):
    return sum(rel / math.log2(i + 2) for i, rel in enumerate(rels[:k]))


def metrics_one(ranked_ids, gold_set, k_recall=5, k_mrr=10, k_ndcg=10):
    gold_set = set(gold_set)

    hit1 = float(ranked_ids[0] in gold_set)
    recall5 = float(any(cid in gold_set for cid in ranked_ids[:k_recall]))

    mrr10 = 0.0
    for rank, cid in enumerate(ranked_ids[:k_mrr], start=1):
        if cid in gold_set:
            mrr10 = 1.0 / rank
            break

    rels = [1.0 if cid in gold_set else 0.0 for cid in ranked_ids[:k_ndcg]]
    dcg = dcg_at_k(rels, k_ndcg)

    ideal_n = min(len(gold_set), k_ndcg)
    idcg = dcg_at_k([1.0] * ideal_n + [0.0] * (k_ndcg - ideal_n), k_ndcg)

    ndcg10 = dcg / idcg if idcg > 0 else 0.0

    return hit1, recall5, mrr10, ndcg10


def summarize(metrics):
    return {
        "n": len(metrics),
        "Hit@1": float(np.mean([m[0] for m in metrics])),
        "Recall@5": float(np.mean([m[1] for m in metrics])),
        "MRR@10": float(np.mean([m[2] for m in metrics])),
        "NDCG@10": float(np.mean([m[3] for m in metrics])),
    }


def benchmark_model_internal(model_name, model_path, queries, corpus, dims=DIMS):
    cids = list(corpus.keys())
    docs = list(corpus.values())

    print("\n" + "#" * 100)
    print(model_name)
    print(model_path)
    print("#" * 100)

    model = SentenceTransformer(model_path)
    model.max_seq_length = 512

    results = {}

    for dim in dims:
        print("\n" + "=" * 80)
        print(f"{model_name} | internal dim={dim}")
        print("=" * 80)

        doc_emb, raw_dim, used_dim = encode_truncated(
            model, docs, dim=dim, batch_size=BATCH_SIZE
        )
        query_emb, _, _ = encode_truncated(
            model,
            [q["question"] for q in queries],
            dim=dim,
            batch_size=BATCH_SIZE,
        )

        metrics = []
        metrics_by_intent = {}

        for qi, q in enumerate(tqdm(queries, desc=f"{model_name} dim={used_dim}")):
            sims = doc_emb @ query_emb[qi]
            order = np.argsort(-sims)
            ranked_ids = [cids[j] for j in order[:10]]

            m = metrics_one(ranked_ids, q["gold"])
            metrics.append(m)

            intent = q.get("intent") or "unknown"
            metrics_by_intent.setdefault(intent, []).append(m)

        result = summarize(metrics)
        result["raw_dim"] = raw_dim
        result["used_dim"] = used_dim
        result["by_intent"] = {k: summarize(v) for k, v in metrics_by_intent.items()}

        results[used_dim] = result

        print(f"raw_dim:   {raw_dim}")
        print(f"used_dim:  {used_dim}")
        print(f"Hit@1:     {result['Hit@1']:.4f}")
        print(f"Recall@5:  {result['Recall@5']:.4f}")
        print(f"MRR@10:    {result['MRR@10']:.4f}")
        print(f"NDCG@10:   {result['NDCG@10']:.4f}")

        print("By intent:")
        for intent, r in result["by_intent"].items():
            print(
                f"  {intent:<10} n={r['n']:<4} "
                f"Hit@1={r['Hit@1']:.4f} "
                f"Recall@5={r['Recall@5']:.4f} "
                f"MRR@10={r['MRR@10']:.4f} "
                f"NDCG@10={r['NDCG@10']:.4f}"
            )

        del doc_emb, query_emb
        gc.collect()
        torch.cuda.empty_cache()

    del model
    gc.collect()
    torch.cuda.empty_cache()

    return results


# ===== Run internal benchmark =====
internal_queries, internal_corpus = load_internal_eval(EVAL_QUERIES, CORPUS_FULL)

print("Internal queries:", len(internal_queries))
print("Internal corpus:", len(internal_corpus))

internal_results = {}

for name, path in MODELS_TO_RUN.items():
    if str(path).startswith("PATH/TO"):
        print("Skip:", name)
        continue

    try:
        internal_results[name] = benchmark_model_internal(
            model_name=name,
            model_path=path,
            queries=internal_queries,
            corpus=internal_corpus,
            dims=DIMS,
        )
    except Exception as e:
        print(f"FAILED {name}: {e}")


print("\n" + "=" * 100)
print("INTERNAL SUMMARY")
print("=" * 100)
print(f"{'Model':<32} {'Dim':>6} {'Hit@1':>8} {'Recall@5':>10} {'MRR@10':>9} {'NDCG@10':>10}")
print("-" * 100)

for name, dim_results in internal_results.items():
    for dim, r in dim_results.items():
        print(
            f"{name:<32} {r['used_dim']:>6} "
            f"{r['Hit@1'] * 100:>7.2f}% "
            f"{r['Recall@5'] * 100:>9.2f}% "
            f"{r['MRR@10']:>9.4f} "
            f"{r['NDCG@10']:>10.4f}"
        )

Internal queries: 158
Internal corpus: 2317

####################################################################################################
BGE-M3 raw
BAAI/bge-m3
####################################################################################################


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]


BGE-M3 raw | internal dim=256


Batches:   0%|          | 0/37 [00:00<?, ?it/s]

Batches:   0%|          | 0/3 [00:00<?, ?it/s]

...

raw_dim:   1024
used_dim:  512
Hit@1:     0.6519
Recall@5:  0.9114
MRR@10:    0.7609
NDCG@10:   0.7595
By intent:
  tra_cuu    n=148  Hit@1=0.6554 Recall@5=0.9189 MRR@10=0.7671 NDCG@10=0.7646
  tinh_toan  n=10   Hit@1=0.6000 Recall@5=0.8000 MRR@10=0.6700 NDCG@10=0.6849

Vietnamese_Embedding_v1 raw | internal dim=1024


Batches:   0%|          | 0/37 [00:00<?, ?it/s]

Batches:   0%|          | 0/3 [00:00<?, ?it/s]

Vietnamese_Embedding_v1 raw dim=1024:   0%|          | 0/158 [00:00<?, ?it/s]

raw_dim:   1024
used_dim:  1024
Hit@1:     0.6709
Recall@5:  0.9304
MRR@10:    0.7832
NDCG@10:   0.7848
By intent:
  tra_cuu    n=148  Hit@1=0.6757 Recall@5=0.9392 MRR@10=0.7908 NDCG@10=0.7916
  tinh_toan  n=10   Hit@1=0.6000 Recall@5=0.8000 MRR@10=0.6700 NDCG@10=0.6849

INTERNAL SUMMARY
Model                               Dim    Hit@1   Recall@5    MRR@10    NDCG@10
----------------------------------------------------------------------------------------------------
BGE-M3 raw                          256   63.92%     87.34%    0.7408     0.7561
BGE-M3 raw                          512   72.15%     91.14%    0.8053     0.8114
BGE-M3 raw                         1024   70.25%     92.41%    0.7989     0.8205
Vietnamese_Embedding_v2 raw         256   57.59%     83.54%    0.6993     0.7028
Vietnamese_Embedding_v2 raw         512   64.56%     89.87%    0.7521     0.7584
Vietnamese_Embedding_v2 raw        1024   67.72%     92.41%    0.7809     0.7923
Baseline trained                    256   6

In [ ]:
import json, gc, random
from pathlib import Path
from collections import defaultdict
import torch
from torch.utils.data import DataLoader
from sentence_transformers import SentenceTransformer, InputExample
from sentence_transformers.losses import MultipleNegativesRankingLoss, MatryoshkaLoss

# ---- 2 bản cần so, mỗi bản 3 seed ----
RUNS = {
    "baseline":  "/content/drive/MyDrive/Data/train_query_positive.jsonl",
    "with_gen":  "/content/drive/MyDrive/Data/train_query_positive_merged_with_gen.jsonl",
}
SEEDS = [42, 123, 7]

BASE_MODEL = "AITeamVN/Vietnamese_Embedding_v2"
OUT_DIR    = "/content/drive/MyDrive/Data/archive/outputs/seed_sweep"

CFG = {
    "epochs": 2, "lr": 2e-6, "batch": 16, "warmup": 10,
    "max_seq_len": 1024, "mrl_dims": [256, 512, 1024], "use_amp": True,
}

def set_seed(seed):
    random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

def load_grouped(path):
    by_q = defaultdict(list)
    with open(path, encoding="utf-8") as f:
        for line in f:
            if not line.strip(): continue
            r = json.loads(line)
            q, pos = str(r.get("query","")).strip(), str(r.get("positive","")).strip()
            if q and pos: by_q[q].append(pos)
    return by_q

def make_examples(by_q, epoch_seed):
    rng = random.Random(epoch_seed)
    ex = [InputExample(texts=[q, rng.choice(poss)]) for q, poss in by_q.items()]
    rng.shuffle(ex)
    return ex

def train_one(train_file, seed, run_name):
    set_seed(seed)                                  # TRƯỚC khi tạo model
    by_q = load_grouped(train_file)

    model = SentenceTransformer(BASE_MODEL)
    model.max_seq_length = CFG["max_seq_len"]
    base_loss = MultipleNegativesRankingLoss(model=model, scale=20.0)
    loss = MatryoshkaLoss(model=model, loss=base_loss, matryoshka_dims=CFG["mrl_dims"])

    ckpt = str(Path(OUT_DIR) / f"{run_name}_seed{seed}")     # <-- tên CHỨA seed
    print(f"\n{'='*60}\n{run_name} | seed={seed} | {len(by_q)} query\n{'='*60}")

    for epoch in range(CFG["epochs"]):
        examples = make_examples(by_q, epoch_seed=seed + epoch)
        loader = DataLoader(examples, batch_size=CFG["batch"], shuffle=True)
        model.fit(
            train_objectives=[(loader, loss)],
            epochs=1,
            warmup_steps=CFG["warmup"] if epoch == 0 else 0,
            optimizer_params={"lr": CFG["lr"]},
            weight_decay=0.01, max_grad_norm=1.0,
            use_amp=CFG["use_amp"] and torch.cuda.is_available(),
            output_path=None, save_best_model=False, show_progress_bar=True,
        )

    model.save(ckpt)
    del model, loss, base_loss
    gc.collect(); torch.cuda.empty_cache()
    print(f"  -> {ckpt}")
    return ckpt

CKPTS = {}
for run_name, train_file in RUNS.items():
    for seed in SEEDS:
        CKPTS[(run_name, seed)] = train_one(train_file, seed, run_name)

print("\n" + "="*60)
print("DONE — 6 checkpoints:")
for (r, s), c in CKPTS.items():
    print(f"  {r:10s} seed={s:3d}: {c}")

/tmp/ipykernel_968/1568673402.py:7: DeprecationWarning: Importing from 'sentence_transformers.losses' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.losses' instead.
  from sentence_transformers.losses import MultipleNegativesRankingLoss, MatryoshkaLoss


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/171 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.45k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

...

Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Step,Training Loss


Step,Training Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  -> /content/drive/MyDrive/Data/archive/outputs/seed_sweep/with_gen_seed7

DONE — 6 checkpoints:
  baseline   seed= 42: /content/drive/MyDrive/Data/archive/outputs/seed_sweep/baseline_seed42
  baseline   seed=123: /content/drive/MyDrive/Data/archive/outputs/seed_sweep/baseline_seed123
  baseline   seed=  7: /content/drive/MyDrive/Data/archive/outputs/seed_sweep/baseline_seed7
  with_gen   seed= 42: /content/drive/MyDrive/Data/archive/outputs/seed_sweep/with_gen_seed42
  with_gen   seed=123: /content/drive/MyDrive/Data/archive/outputs/seed_sweep/with_gen_seed123
  with_gen   seed=  7: /content/drive/MyDrive/Data/archive/outputs/seed_sweep/with_gen_seed7


In [ ]:
import json, gc, os, glob
import numpy as np
import torch
from sentence_transformers import SentenceTransformer

SWEEP_DIR = "/content/drive/MyDrive/Data/archive/outputs/seed_sweep"
RUNS  = ["baseline", "with_gen"]
SEEDS = [42, 123, 7]

EVAL_SETS = {
    "INTERNAL": {
        "query_file": "/content/drive/MyDrive/Data/eval_queries.jsonl",
        "corpus":     "/content/drive/MyDrive/Data/corpus_full.jsonl",
        "format": "jsonl",
        "f_query": "query", "f_gold": "gold_chunk_ids",
    },
    "EXTERNAL": {
        "query_file": "/content/drive/MyDrive/Data/question.json",
        "corpus":     "/content/drive/MyDrive/Data/chunk_outputs1_finals",
        "format": "json",
        "f_query": "question", "f_gold": "gold_chunk_ids",
    },
}
C_ID, C_TEXT = "chunk_id", "text"
DIMS = [256, 512, 1024]
MAX_SEQ = 1024

def load_corpus(src):
    c = {}
    if os.path.isfile(src):
        with open(src, encoding="utf-8") as f:
            for line in f:
                if line.strip():
                    r = json.loads(line); c[r[C_ID]] = r[C_TEXT]
        return c

    for fp in sorted(glob.glob(os.path.join(src, "**", "*.json"), recursive=True)):
        with open(fp, encoding="utf-8") as f:
            arr = json.load(f)
        for item in arr:
            md = item.get("metadata", {})
            doc_id = md.get("document_id")
            idx    = md.get("chunk_index")
            text   = item.get("page_content", "")
            if doc_id is None or idx is None or not text.strip():
                continue
            cid = f"{doc_id}::chunk::{idx}"
            c[cid] = text
    return c

def load_eval(cfg, corpus):
    if cfg.get("format") == "json":
        with open(cfg["query_file"], encoding="utf-8") as f:
            rows = json.load(f)
    else:
        rows = []
        with open(cfg["query_file"], encoding="utf-8") as f:
            for line in f:
                if line.strip(): rows.append(json.loads(line))

    Q, R, miss = {}, {}, 0
    for i, r in enumerate(rows):
        gold_all = r.get(cfg["f_gold"]) or []
        gold = {g for g in gold_all if g in corpus}
        if gold_all and not gold: miss += 1
        if not gold: continue
        qid = f"q{i:04d}"
        Q[qid] = r[cfg["f_query"]]
        R[qid] = gold
    return Q, R, miss

def metrics(model, Q, R, corpus, dim):
    cids = list(corpus.keys())
    c_emb = model.encode([corpus[c] for c in cids], normalize_embeddings=True,
                        batch_size=64, show_progress_bar=False)[:, :dim]
    c_emb /= (np.linalg.norm(c_emb, axis=1, keepdims=True) + 1e-9)   # renorm sau truncate
    qids = list(Q.keys())
    q_emb = model.encode([Q[q] for q in qids], normalize_embeddings=True,
                        batch_size=64, show_progress_bar=False)[:, :dim]
    q_emb /= (np.linalg.norm(q_emb, axis=1, keepdims=True) + 1e-9)

    sims = q_emb @ c_emb.T
    h1 = r5 = mrr = ndcg = 0.0
    for i, qid in enumerate(qids):
        ranked = [cids[j] for j in np.argsort(-sims[i])]
        gold = R[qid]
        if ranked[0] in gold: h1 += 1
        if any(c in gold for c in ranked[:5]): r5 += 1
        for rk, c in enumerate(ranked[:10], 1):
            if c in gold: mrr += 1.0/rk; break
        dcg = sum(1.0/np.log2(rk+1) for rk, c in enumerate(ranked[:10],1) if c in gold)
        idcg = sum(1.0/np.log2(rk+1) for rk in range(1, min(len(gold),10)+1))
        ndcg += dcg/idcg if idcg else 0
    n = len(qids)
    return dict(hit1=h1/n, rec5=r5/n, mrr=mrr/n, ndcg=ndcg/n)

EVALS = {}
for ename, cfg in EVAL_SETS.items():
    corpus = load_corpus(cfg["corpus"])
    Q, R, miss = load_eval(cfg, corpus)
    EVALS[ename] = (Q, R, corpus)
    print(f"{ename}: {len(Q)} query | corpus {len(corpus)} | gold KHÔNG khớp corpus: {miss}")
    if miss > 0:
        print(f"{miss} query gold không có trong corpus — KIỂM ID MAPPING trước khi tin số!")

# Bench 6 checkpoint
results = {}   # (run, seed, eval, dim) -> metrics
for run in RUNS:
    for seed in SEEDS:
        path = f"{SWEEP_DIR}/{run}_seed{seed}"
        model = SentenceTransformer(path); model.max_seq_length = MAX_SEQ
        print(f"\nBench {run} seed{seed}...")
        for ename, (Q, R, corpus) in EVALS.items():
            for dim in DIMS:
                results[(run, seed, ename, dim)] = metrics(model, Q, R, corpus, dim)
        del model; gc.collect(); torch.cuda.empty_cache()

# ===== BẢNG mean ± std =====
for ename in EVAL_SETS:
    print(f"\n{'='*78}\n{ename} — mean ± std trên {len(SEEDS)} seed\n{'='*78}")
    print(f"{'Run':<10} {'Dim':>5} {'Hit@1':>16} {'Recall@5':>16} {'MRR@10':>16} {'NDCG@10':>16}")
    print("-"*78)
    for run in RUNS:
        for dim in DIMS:
            vals = [results[(run, s, ename, dim)] for s in SEEDS]
            def ms(k, pct=True):
                a = np.array([v[k] for v in vals])
                if pct: return f"{a.mean()*100:6.2f}±{a.std()*100:4.2f}%"
                return f"{a.mean():6.4f}±{a.std():5.4f}"
            print(f"{run:<10} {dim:>5} {ms('hit1'):>16} {ms('rec5'):>16} "
                  f"{ms('mrr',False):>16} {ms('ndcg',False):>16}")
        print()

print("="*78)
print("PHÂN ĐỊNH (dim 1024, Hit@1): khoảng mean±std có CHỒNG nhau không?")
print("="*78)
for ename in EVAL_SETS:
    a = np.array([results[("baseline", s, ename, 1024)]["hit1"] for s in SEEDS])
    b = np.array([results[("with_gen", s, ename, 1024)]["hit1"] for s in SEEDS])
    lo_a, hi_a = a.mean()-a.std(), a.mean()+a.std()
    lo_b, hi_b = b.mean()-b.std(), b.mean()+b.std()
    overlap = not (hi_a < lo_b or hi_b < lo_a)
    verdict = "CHỒNG → khác biệt nằm trong nhiễu, KHÔNG kết luận được" if overlap \
              else ("TÁCH → with_gen THẮNG thật" if b.mean() > a.mean() else "TÁCH → with_gen KÉM thật")
    print(f"{ename:9s} baseline {a.mean()*100:.2f}±{a.std()*100:.2f}  |  "
          f"with_gen {b.mean()*100:.2f}±{b.std()*100:.2f}  →  {verdict}")

INTERNAL: 158 query | corpus 2317 | gold KHÔNG khớp corpus: 0
EXTERNAL: 390 query | corpus 813 | gold KHÔNG khớp corpus: 0


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]


Bench baseline seed42...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]


Bench baseline seed123...


...


Bench with_gen seed42...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]


Bench with_gen seed123...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]


Bench with_gen seed7...

INTERNAL — mean ± std trên 3 seed
Run          Dim            Hit@1         Recall@5           MRR@10          NDCG@10
------------------------------------------------------------------------------
baseline     256      62.45±1.08%      89.45±0.79%    0.7489±0.0055    0.7646±0.0021
baseline     512      70.46±0.30%      93.67±0.00%    0.8028±0.0020    0.8149±0.0016
baseline    1024      73.21±0.60%      94.73±0.30%    0.8220±0.0037    0.8269±0.0031

with_gen     256      65.61±1.08%      90.93±0.60%    0.7671±0.0057    0.7798±0.0032
with_gen     512      70.04±0.79%      94.09±0.30%    0.7987±0.0060    0.8123±0.0048
with_gen    1024      71.52±0.52%      94.73±0.60%    0.8147±0.0025    0.8245±0.0018


EXTERNAL — mean ± std trên 3 seed
Run          Dim            Hit@1         Recall@5           MRR@10          NDCG@10
------------------------------------------------------------------------------
baseline     256      68.38±0.12%      88.29±0.32%    0.7711±0.00

In [ ]:
import json, gc, os, glob, math
import numpy as np
import torch
from sentence_transformers import SentenceTransformer

# ================== CONFIG ==================
RAW_MODELS = {
    "BGE-M3 raw":                  "BAAI/bge-m3",
    "Vietnamese_Embedding_v1 raw": "AITeamVN/Vietnamese_Embedding",
    "Vietnamese_Embedding_v2 raw": "AITeamVN/Vietnamese_Embedding_v2",
}

EVAL_SETS = {
    "D1 (Internal)": {
        "query_file": "/content/drive/MyDrive/Data/eval_queries.jsonl",
        "corpus":     "/content/drive/MyDrive/Data/corpus_full.jsonl",
        "format": "jsonl", "f_query": "query", "f_gold": "gold_chunk_ids",
    },
    "D2 (External)": {
        "query_file": "/content/drive/MyDrive/Data/question.json",
        "corpus":     "/content/drive/MyDrive/Data/chunk_outputs1_finals",
        "format": "json",  "f_query": "question", "f_gold": "gold_chunk_ids",
    },
}

C_ID, C_TEXT = "chunk_id", "text"
DIMS    = [256, 512, 1024]
MAX_SEQ = 1024          # ← PHẢI giống lúc đo fine-tuned (seed sweep)
BATCH   = 64
# ============================================


def load_corpus(src):
    c = {}
    if os.path.isfile(src):                       # D1: file .jsonl
        with open(src, encoding="utf-8") as f:
            for line in f:
                if line.strip():
                    r = json.loads(line); c[r[C_ID]] = r[C_TEXT]
        return c
    for fp in sorted(glob.glob(os.path.join(src, "**", "*.json"), recursive=True)):
        with open(fp, encoding="utf-8") as f:     # D2: 49 folder × JSON array
            arr = json.load(f)
        for item in arr:
            md = item.get("metadata", {})
            doc_id, idx = md.get("document_id"), md.get("chunk_index")
            text = item.get("page_content", "")
            if doc_id is None or idx is None or not text.strip():
                continue
            c[f"{doc_id}::chunk::{idx}"] = text
    return c


def load_eval(cfg, corpus):
    if cfg["format"] == "json":
        rows = json.load(open(cfg["query_file"], encoding="utf-8"))
    else:
        rows = [json.loads(l) for l in open(cfg["query_file"], encoding="utf-8") if l.strip()]
    Q, R, miss = {}, {}, 0
    for i, r in enumerate(rows):
        gold_all = r.get(cfg["f_gold"]) or []
        gold = {g for g in gold_all if g in corpus}
        if gold_all and not gold: miss += 1
        if not gold: continue
        qid = f"q{i:04d}"
        Q[qid] = r[cfg["f_query"]]; R[qid] = gold
    return Q, R, miss


def metrics(model, Q, R, corpus, dim):
    cids = list(corpus.keys())
    c_emb = model.encode([corpus[c] for c in cids], normalize_embeddings=True,
                         batch_size=BATCH, show_progress_bar=False)[:, :dim]
    c_emb = c_emb / np.maximum(np.linalg.norm(c_emb, axis=1, keepdims=True), 1e-12)
    qids = list(Q.keys())
    q_emb = model.encode([Q[q] for q in qids], normalize_embeddings=True,
                         batch_size=BATCH, show_progress_bar=False)[:, :dim]
    q_emb = q_emb / np.maximum(np.linalg.norm(q_emb, axis=1, keepdims=True), 1e-12)

    sims = q_emb @ c_emb.T
    h1 = r5 = mrr = ndcg = 0.0
    for i, qid in enumerate(qids):
        ranked = [cids[j] for j in np.argsort(-sims[i])]
        gold = R[qid]
        if ranked[0] in gold: h1 += 1
        if any(c in gold for c in ranked[:5]): r5 += 1
        for rk, c in enumerate(ranked[:10], 1):
            if c in gold: mrr += 1.0/rk; break
        dcg  = sum(1.0/math.log2(rk+1) for rk, c in enumerate(ranked[:10], 1) if c in gold)
        idcg = sum(1.0/math.log2(rk+1) for rk in range(1, min(len(gold), 10)+1))
        ndcg += dcg/idcg if idcg else 0
    n = len(qids)
    return dict(hit1=h1/n, rec5=r5/n, mrr=mrr/n, ndcg=ndcg/n)


# ---------- Load eval sets ----------
EVALS = {}
for ename, cfg in EVAL_SETS.items():
    corpus = load_corpus(cfg["corpus"])
    Q, R, miss = load_eval(cfg, corpus)
    EVALS[ename] = (Q, R, corpus)
    print(f"{ename}: {len(Q)} query | corpus {len(corpus)} | gold không khớp: {miss}")
    if miss: print(f"  ⚠️  KIỂM ID MAPPING!")

# ---------- Bench 3 baseline raw ở MAX_SEQ=1024 ----------
print(f"\n{'='*80}\nBASELINE RAW — đo lại ở max_seq_length={MAX_SEQ} (giống lúc đo fine-tuned)\n{'='*80}")

results = {}
for name, path in RAW_MODELS.items():
    model = SentenceTransformer(path)
    model.max_seq_length = MAX_SEQ                      # ← điểm mấu chốt
    print(f"\n### {name}  (dim gốc: {model.get_sentence_embedding_dimension()})")
    for ename, (Q, R, corpus) in EVALS.items():
        print(f"  {ename}")
        print(f"  {'Dim':>5} {'Acc@1':>8} {'Acc@5':>8} {'MRR@10':>9} {'NDCG@10':>9}")
        print("  " + "-"*44)
        for dim in DIMS:
            m = metrics(model, Q, R, corpus, dim)
            results[(name, ename, dim)] = m
            print(f"  {dim:>5} {m['hit1']*100:>7.2f}% {m['rec5']*100:>7.2f}% "
                  f"{m['mrr']:>9.4f} {m['ndcg']:>9.4f}")
    del model; gc.collect(); torch.cuda.empty_cache()

# ---------- Bảng gọn 1024d để dán vào quyển ----------
print(f"\n{'='*80}\nTÓM TẮT 1024d (dán vào bảng 4.25 / 4.26)\n{'='*80}")
for ename in EVAL_SETS:
    print(f"\n{ename}")
    print(f"{'Model':<32} {'Acc@1':>8} {'Acc@5':>8} {'MRR@10':>9} {'NDCG@10':>9}")
    print("-"*70)
    for name in RAW_MODELS:
        m = results[(name, ename, 1024)]
        print(f"{name:<32} {m['hit1']*100:>7.2f}% {m['rec5']*100:>7.2f}% "
              f"{m['mrr']:>9.4f} {m['ndcg']:>9.4f}")

D1 (Internal): 158 query | corpus 2317 | gold không khớp: 0
D2 (External): 390 query | corpus 813 | gold không khớp: 0

BASELINE RAW — đo lại ở max_seq_length=1024 (giống lúc đo fine-tuned)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/297 [00:00<?, ?B/s]

/tmp/ipykernel_3394/3747156900.py:112: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"\n### {name}  (dim gốc: {model.get_sentence_embedding_dimension()})")



### Vietnamese_Embedding_v2 raw  (dim gốc: 1024)
  D1 (Internal)
    Dim    Acc@1    Acc@5    MRR@10   NDCG@10
  --------------------------------------------
    256   58.23%   82.91%    0.6947    0.6907
    512   60.76%   87.34%    0.7240    0.7354
   1024   65.19%   91.77%    0.7694    0.7800
  D2 (External)
    Dim    Acc@1    Acc@5    MRR@10   NDCG@10
  --------------------------------------------
    256   62.56%   86.67%    0.7297    0.7715
    512   69.74%   89.49%    0.7799    0.8164
   1024   70.77%   91.54%    0.7951    0.8315

TÓM TẮT 1024d (dán vào bảng 4.25 / 4.26)

D1 (Internal)
Model                               Acc@1    Acc@5    MRR@10   NDCG@10
----------------------------------------------------------------------
BGE-M3 raw                         70.89%   92.41%    0.8015    0.8207
Vietnamese_Embedding_v1 raw        65.82%   93.04%    0.7718    0.7753
Vietnamese_Embedding_v2 raw        65.19%   91.77%    0.7694    0.7800

D2 (External)
Model                        